# Web Scraping Using Selenium

So far we've scraped static HTML (BeautifulSoup) and structured JSON (APIs). Some websites, though, load their data *after* the page first loads, using JavaScript to fetch and render content dynamically. If you try to scrape these with `requests` or `urllib`, you'll just get an empty shell of a page, because the data hasn't loaded yet.

Selenium solves this by actually driving a real browser: it clicks buttons, fills in forms, waits for content to load, and then lets us read the page exactly as a human would see it.

In this lab, we'll scrape historical oil prices by province from PTT's website, which loads its data this way.

### References

- https://www.selenium.dev/documentation/webdriver/
- https://www.selenium.dev/documentation/support_packages/working_with_select_elements/
- https://selenium-python.readthedocs.io/getting-started.html#simple-usage
- https://www.guru99.com/xpath-selenium.html


### Setup

In [ ]:
# Install required libraries
#!pip install selenium
#!pip install webdriver-manager

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.select import Select
import pandas as pd
import time

/Users/anushkaojha/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Set up the Chrome driver
service = Service(ChromeDriverManager().install())
wd = webdriver.Chrome(service=service)

### What we're about to do on this page

Let's look at what's actually on PTT's oil price page (`pttor.com/en/oil_price`), since that's what we're about to automate.

**Page layout:**
- A section for searching historical oil prices, with a set of filters: a **province/district dropdown**, a **month dropdown**, and a **year dropdown**
- A **search button** that applies whatever filters you've selected
- Below that, a **results table** that shows fuel prices for the province/date you searched, but this table doesn't exist on the page until you actually click search. The page shows a loading message ("Loading oil price, please wait...") while it fetches the data behind the scenes.

<img src="./images/webpage.png" align="center" style="width:1200px;"/>

This table's data is loaded dynamically, *after* the initial page load, using JavaScript, which is exactly why a tool like BeautifulSoup (which only sees the page as it was at the moment it was first downloaded) wouldn't work here. 

Selenium can wait for that JavaScript to finish and the table to actually appear, because it's controlling a real browser instead of just downloading raw HTML.

**Our plan for automating it:**
1. **Open the page** in a real Chrome browser
2. **Select a province** from the dropdown. 
3. **Select a month and year** from their own dropdowns
4. **Click the search button** to apply those filters
5. **Read the resulting table** and turn it into a pandas DataFrame, the same way we did with BeautifulSoup

Each of these steps means finding a specific element on the page, a dropdown, a button, a table, and telling Selenium exactly where it is. 

In [3]:
# Open the webpage
url = "https://www.pttor.com/en/oil_price"
wd.get(url)
print(f"Successfully opened: {url}\n")
time.sleep(3)  # Wait for the page to fully load

Successfully opened: https://www.pttor.com/en/oil_price



## What is XPath?

XPath stands for XML Path Language. It's a way of writing a "path" to a specific element in an HTML document's tree structure, similar to how a file path (`/folder/subfolder/file.txt`) locates a file.

Selenium uses XPath (among other methods) to find elements on a page so it can click them, read their text, or type into them.

### How to find elements using XPath in Selenium

The easiest way to find the right XPath for an element is:
1. Right-click the element on the page → **Inspect**
2. In DevTools, right-click the highlighted HTML → **Copy → Copy XPath**
3. Paste that into `wd.find_element(By.XPATH, "...")` below

In [4]:
province_filter = wd.find_element(By.XPATH, '//*[@id="province_history"]')
province_object = Select(province_filter)
province_object.select_by_visible_text("Nonthaburi")
time.sleep(2)

In [5]:
month_filter = wd.find_element(By.XPATH, '//*[@id="month_history"]')
month_object = Select(month_filter)
month_object.select_by_visible_text("January")
time.sleep(2)

In [6]:
# TODO: Inspect the page and replace this with the actual XPath for the
# year dropdown filter
year_filter = wd.find_element(By.XPATH, '//*[@id="year_history"]')
year_object = Select(year_filter)
year_object.select_by_visible_text("2025")
time.sleep(2)

In [7]:
# TODO: Inspect the page and replace this with the actual XPath for the
# search/submit button that applies the filters
search_button = wd.find_element(By.XPATH, '//*[@id="jet-tabs-content-6882"]/div/div/div/div/div/div[1]/div[1]/button')
search_button.click()
time.sleep(5)

### Reading the results table

Now that the filters are applied, let's locate the results table and pull its data out.

In [8]:
# results table (it will contain the oil price rows for the filters you set)
results_table = wd.find_element(By.XPATH, '//*[@id="historical_table"]')

In [9]:
# Preview the raw text of the table, to sanity-check we found the right element
results_table.text

'Date - Time\n23-01-2568 05:00 0.00 32.94 33.54 35.38 35.75 44.04 44.34 44.94 0.00\n15-01-2568 05:00 0.00 32.94 33.94 35.78 36.15 44.44 44.74 44.94 0.00\n11-01-2568 05:00 0.00 32.94 33.44 35.28 35.65 43.94 44.24 44.94 0.00\n03-01-2568 05:00 0.00 32.94 33.84 35.58 35.95 44.24 44.54 44.94 0.00'

`results_table`  is just one single element. The whole <table> as a block. 

In [10]:
# Find all the rows (<tr> tags) inside the table's body
table_rows = results_table.find_elements(By.TAG_NAME, "tr")
print(f"Number of rows found: {len(table_rows)}")

Number of rows found: 5


In Selenium, `find_element` (singular) gives you just the first match, while `find_elements` (plural) gives you all matches, as a list.

In [11]:
# Loop through each row and print its cell contents
for row in table_rows:
    cells = row.find_elements(By.TAG_NAME, "td")
    for cell in cells:
        print(cell.text, end="\t")
    print()


23-01-2568 05:00	0.00	32.94	33.54	35.38	35.75	44.04	44.34	44.94	0.00	
15-01-2568 05:00	0.00	32.94	33.94	35.78	36.15	44.44	44.74	44.94	0.00	
11-01-2568 05:00	0.00	32.94	33.44	35.28	35.65	43.94	44.24	44.94	0.00	
03-01-2568 05:00	0.00	32.94	33.84	35.58	35.95	44.24	44.54	44.94	0.00	


### Saving the table data

Printing the rows was just for us to visually check the data looked right. But printed text isn't something we can actually *use*, we can't filter it, sort it, or turn it into a DataFrame. 

In [12]:
# Save each row's cell values into a list of lists
row_list = []
for row in table_rows:
    oil_item = []
    for cell in row.find_elements(By.TAG_NAME, "td"):
        oil_item.append(cell.text)
    row_list.append(oil_item)

row_list

[[],
 ['23-01-2568 05:00',
  '0.00',
  '32.94',
  '33.54',
  '35.38',
  '35.75',
  '44.04',
  '44.34',
  '44.94',
  '0.00'],
 ['15-01-2568 05:00',
  '0.00',
  '32.94',
  '33.94',
  '35.78',
  '36.15',
  '44.44',
  '44.74',
  '44.94',
  '0.00'],
 ['11-01-2568 05:00',
  '0.00',
  '32.94',
  '33.44',
  '35.28',
  '35.65',
  '43.94',
  '44.24',
  '44.94',
  '0.00'],
 ['03-01-2568 05:00',
  '0.00',
  '32.94',
  '33.84',
  '35.58',
  '35.95',
  '44.24',
  '44.54',
  '44.94',
  '0.00']]

Let's check the first row, since the headers are images, we'd expect this to come back empty or malformed.

In [13]:
row_list[0]

[]

In [15]:
# Remove the first row because it's empty (headers are images, not text)
row_list.pop(0)
row_list

[['15-01-2568 05:00',
  '0.00',
  '32.94',
  '33.94',
  '35.78',
  '36.15',
  '44.44',
  '44.74',
  '44.94',
  '0.00'],
 ['11-01-2568 05:00',
  '0.00',
  '32.94',
  '33.44',
  '35.28',
  '35.65',
  '43.94',
  '44.24',
  '44.94',
  '0.00'],
 ['03-01-2568 05:00',
  '0.00',
  '32.94',
  '33.84',
  '35.58',
  '35.95',
  '44.24',
  '44.54',
  '44.94',
  '0.00']]

Now let's assign column names manually, since we can't pull them from the table itself.

In [16]:
# The table's actual header images correspond to these fuel types, in order:
# Diesel B20, Diesel, Gasohol E20, Gasohol 91, Gasohol 95, Gasoline 95,
# Super Power GSH95, Premium Diesel, Super Power X99

oil_columns = ["DateTime", "DieselB20", "Diesel", "GasoholE20", "Gasohol91",
               "Gasohol95", "Gasoline95", "SuperPowerGSH95", "PremiumDiesel", "SuperPowerX99"]

In [17]:
df = pd.DataFrame(row_list, columns=oil_columns)
df

,DateTime,DieselB20,Diesel,GasoholE20,Gasohol91,Gasohol95,Gasoline95,SuperPowerGSH95,PremiumDiesel,SuperPowerX99
0,15-01-2568 05:00,0.00,32.94,33.94,35.78,36.15,44.44,44.74,44.94,0.00
1,11-01-2568 05:00,0.00,32.94,33.44,35.28,35.65,43.94,44.24,44.94,0.00
2,03-01-2568 05:00,0.00,32.94,33.84,35.58,35.95,44.24,44.54,44.94,0.00


In [18]:
wd.quit()